# Postcards from my jungle: an animal language and 16 sounds

CART 498 · Assignment 5 · Dan Truong

Two pre-trained models, chained together:

1. **Language.** GPT-4.1 (OpenAI Responses API) invents the animal language from a **structured prompt** and writes 16 creatures:
   each has a name, a phrase in the language, a translation, and a description of its voice.
2. **Sound.** Each voice description becomes the prompt for **Stable Audio Open 1.0** (diffusers `StableAudioPipeline`, from the class notebook),
   which makes 16 sounds with different durations.

So the sound of each creature comes from the same text as its words.

**Before running:** GPU runtime; Colab secrets `OPENAI_API_Key` and `HF_TOKEN` with notebook access;
accept the terms of `stabilityai/stable-audio-open-1.0` on Hugging Face (the model is gated).

## 0. Setup

In [ ]:
# Same install as the class notebook
!pip uninstall -y -q torchao
!pip install -q torchsde -U diffusers openai

In [ ]:
import json, os, re
import torch
import soundfile as sf
from IPython.display import Audio, display
from google.colab import userdata
from huggingface_hub import login
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(api_key=userdata.get("OPENAI_API_Key"))
login(token=userdata.get("HF_TOKEN"))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

## 1. The animal language: a structured prompt

The prompt has four parts: **role**, **sound system**, **grammar**, and **output**.
The rules make the language consistent (the same sounds and endings come back), instead of random letters.
The output is a schema (Pydantic + structured outputs), so I always get exactly the fields the website needs.

In [ ]:
LANGUAGE_PROMPT = """ROLE
You are a field linguist in an imaginary jungle where tiny, soft, fairy-like creatures live
(fluffy faces, big eyes, butterfly wings). You document their language, called Liluvé.

SOUND SYSTEM (use only these)
- Consonants: l, m, n, v, f, s, h, p, r (rolled "rr" only for strong feelings).
- Vowels: a, e, i, o, u, and the long, sung vowels ā, ē, ī, ō, ū.
- Syllables are open (they end in a vowel). No more than 3 syllables in a word.
- Soft and cute: no harsh sounds (no k, g, t, d, x, z).

GRAMMAR
- Word order: verb first, then who, then what ("sing I moon").
- Ending -i makes a word small and dear ("mo" = flower, "moi" = little flower).
- Repeating a syllable makes a plural or something that keeps happening ("fafa" = many wings).
- Every phrase ends with a call that matches the creature's mood: "-ō" (happy), "-ū" (sleepy), "-ē" (alarm).
- Reuse the same core words across creatures (e.g. "lu" = light, "mo" = flower, "fa" = wing, "sī" = rain), so the 16 phrases read like one language.

OUTPUT
Write exactly 16 creatures. For each one:
- name: in Liluvé, 2-3 syllables.
- phrase: what it says, 3 to 7 words of Liluvé, following the rules.
- translation: a short, sweet English translation.
- voice: how it sounds, written for a sound-generation model: a concrete description of an animal call
  (pitch, rhythm, texture, comparisons to real animals or objects), no words, no music, no instruments.
- duration_s: how long the sound lasts, between 1.5 and 10 seconds. Make the durations clearly different from each other.
Also give the language's name and the list of core words you used."""

class Creature(BaseModel):
    name: str
    phrase: str
    translation: str
    voice: str = Field(description="Description of the creature's call for a text-to-audio model")
    duration_s: float

class CoreWord(BaseModel):
    word: str
    meaning: str

class Bestiary(BaseModel):
    language: str
    core_words: list[CoreWord]
    creatures: list[Creature]

response = client.responses.parse(
    model="gpt-4.1",
    input=[{"role": "developer", "content": LANGUAGE_PROMPT},
           {"role": "user", "content": "Document the 16 creatures."}],
    text_format=Bestiary,
    temperature=1.0,
)
bestiary = response.output_parsed
assert len(bestiary.creatures) == 16, f"got {len(bestiary.creatures)} creatures"
for c in bestiary.creatures:
    c.duration_s = round(min(max(c.duration_s, 1.5), 10.0), 1)    # keep inside the range the site expects

print("Language:", bestiary.language)
print("Core words:", ", ".join(f"{w.word} = {w.meaning}" for w in bestiary.core_words))
print()
for i, c in enumerate(bestiary.creatures, 1):
    print(f"{i:2}. {c.name}  ({c.duration_s} s)\n    {c.phrase}\n    \"{c.translation}\"\n    voice: {c.voice}\n")

In [ ]:
# A quick check that the language follows its own sound rules
import re
forbidden = re.compile(r"[ktdgxz]", re.IGNORECASE)
for c in bestiary.creatures:
    bad = forbidden.findall(c.name + " " + c.phrase)
    if bad:
        print(f"{c.name}: uses forbidden letters {sorted(set(bad))}")
print("Durations:", sorted(c.duration_s for c in bestiary.creatures))

## 2. Sounds with Stable Audio Open

The model is loaded once, then called 16 times. Each prompt is the creature's **voice** description plus a short style note,
and `audio_end_in_s` uses the creature's own duration. The negative prompt pushes away music and speech.

In [ ]:
from diffusers import StableAudioPipeline

pipe = StableAudioPipeline.from_pretrained("stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16).to(device)
sr = pipe.vae.sampling_rate
print("sample rate:", sr)

In [ ]:
STYLE = "Close-up field recording of a small, cute creature in a quiet rainforest at dawn. Natural animal call, clean, no music."
NEGATIVE = "music, melody, drums, instruments, human speech, singing voice, low quality, distortion, noise"

os.makedirs("sounds", exist_ok=True)
sound_files = []

def slug(name):
    return re.sub(r"[^a-z]+", "-", name.lower().replace("ā", "a").replace("ē", "e").replace("ī", "i").replace("ō", "o").replace("ū", "u")).strip("-")

for i, c in enumerate(bestiary.creatures, 1):
    generator = torch.Generator(device).manual_seed(100 + i)
    audio = pipe(
        prompt=f"{c.voice} {STYLE}",
        negative_prompt=NEGATIVE,
        num_inference_steps=100,
        audio_end_in_s=c.duration_s,
        num_waveforms_per_prompt=1,
        generator=generator,
    ).audios[0]
    audio = audio / audio.abs().max().clamp(min=1e-8) * 0.9          # same loudness for every creature
    path = f"sounds/{i:02d}-{slug(c.name)}.wav"
    sf.write(path, audio.T.float().cpu().numpy(), sr)
    sound_files.append(path)
    print(f"{i:2}. {c.name}: {c.duration_s} s → {path}")
    display(Audio(path))

## 3. Save everything for the website

`creatures.json` holds, for each creature, its name, phrase, translation and sound file. The website reads it.

In [ ]:
import zipfile
from google.colab import files, _message

data = {
    "language": bestiary.language,
    "core_words": [w.model_dump() for w in bestiary.core_words],
    "creatures": [{**c.model_dump(), "sound": path} for c, path in zip(bestiary.creatures, sound_files)],
}
with open("creatures.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# Save this notebook, with its outputs, and zip it with the sounds
nb = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]
with open("sound-generation.ipynb", "w") as f:
    json.dump(nb, f)

with zipfile.ZipFile("jungle_sounds.zip", "w") as z:
    for name in sorted(os.listdir("sounds")):
        z.write(f"sounds/{name}")
    z.write("creatures.json")
    z.write("sound-generation.ipynb")
print(round(os.path.getsize("jungle_sounds.zip") / 1e6, 1), "MB")
files.download("jungle_sounds.zip")